In [ ]:
pip install pandas pyfaidx Bio pyarrow fastparquet

In [ ]:
import pandas as pd
import subprocess
import os
import sys
import math
from io import StringIO
import warnings
import pyarrow as pa
import pyarrow.parquet as pq
warnings.filterwarnings('ignore')

# ==============================
# 1. CONFIGURATIONS
# ==============================
INPUT_PARQUET = r"D:/variant_data/clinvar_clean.parquet"
CHUNK_SIZE = 50000

# Thư mục chứa dữ liệu trên máy host (sẽ được mount vào Docker)
WORKDIR = r"D:/variant_data"
CACHE_DIR = r"D:/vep_cache" 
RESOURCES_DIR = r"D:/vep_resources" # Nơi chứa các file .bw, .vcf.gz cho plugins

ASSEMBLY = "GRCh38"
FASTA_FILENAME = "Homo_sapiens.GRCh38.dna.primary_assembly.fa" 

# Đường dẫn mount trên Docker
DOCKER_WORKDIR = "/input"
DOCKER_CACHE = "/opt/vep/.vep"
DOCKER_RESOURCES = "/opt/vep/resources"

# File tạm
TMP_VCF = os.path.join(WORKDIR, "tmp_input.vcf")
TMP_VEP_OUT = os.path.join(WORKDIR, "tmp_output.txt")

# ==============================
# 2. HELPER FUNCTIONS
# ==============================
def process_chunk(df_chunk, chunk_idx):
    """Xử lý một batch VCF qua VEP Docker và trả về DataFrame đã merge."""
    
    df_chunk["CHROM"] = df_chunk["CHROM"].astype(str).str.replace(r"^chr", "", regex=True, case=False)
    # Tạo ID duy nhất để đảm bảo merge chính xác 100%
    df_chunk["Merge_ID"] = (
        df_chunk["CHROM"].astype(str) + "_" +
        df_chunk["POS"].astype(str) + "_" +
        df_chunk["REF"].astype(str) + "/" +
        df_chunk["ALT"].astype(str)
    )

    # 2.1 Ghi file VCF tạm
    with open(TMP_VCF, "w", newline="") as f:
        f.write("##fileformat=VCFv4.2\n")
        f.write("#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n")
        for _, row in df_chunk.iterrows():
            f.write(f"{row['CHROM']}\t{row['POS']}\t{row['Merge_ID']}\t{row['REF']}\t{row['ALT']}\t.\t.\t.\n")

    # 2.2 Cấu hình lệnh chạy VEP với đầy đủ Plugins
    workdir_mnt = os.path.abspath(WORKDIR).replace('\\', '/')
    cache_mnt = os.path.abspath(CACHE_DIR).replace('\\', '/')
    resources_mnt = os.path.abspath(RESOURCES_DIR).replace('\\', '/')

    vep_cmd = [
        "docker", "run", "--rm",
        "-v", f"{workdir_mnt}:{DOCKER_WORKDIR}",
        "-v", f"{cache_mnt}:{DOCKER_CACHE}",
        "-v", f"{resources_mnt}:{DOCKER_RESOURCES}",
        "ensemblorg/ensembl-vep",
        "vep",
        "-i", f"{DOCKER_WORKDIR}/{os.path.basename(TMP_VCF)}",
        "-o", f"{DOCKER_WORKDIR}/{os.path.basename(TMP_VEP_OUT)}",
        "--assembly", ASSEMBLY,
        "--cache", "--offline",
        "--fasta", f"{DOCKER_RESOURCES}/{FASTA_FILENAME}",
        "--tab", "--force_overwrite",
        "--fork", "4",
        
        # Tiêu chí chọn transcript
        "--mane_select", 
        "--canonical",
        "--protein",
        "--hgvs",
        "--pick",
        "--symbol",
        
        # Các cờ thông tin & Tần số quần thể
        "--af_1kg",
        "--af_gnomade",
        "--regulatory",
        
        # Plugins (Đảm bảo file đã có trong thư mục resources)
        "--plugin", f"SpliceAI,snv={DOCKER_RESOURCES}/spliceai_scores.raw.snv.ensembl_mane_v1.4.grch38.vcf.gz",
        "--plugin", (
            f"dbNSFP,{DOCKER_RESOURCES}/dbNSFP5.3.1a_grch38.gz," 
            "phyloP100way_vertebrate,phyloP470way_mammalian,phyloP17way_primate,"
            "phastCons100way_vertebrate,phastCons470way_mammalian,phastCons17way_primate,"
            "GERP++_RS,GERP++_NR,GERP_92_mammals"
        ),
        
        # Chỉ định các cột đầu ra (Thêm các trường từ Plugin)
        "--fields", (
            "Uploaded_variation,Location,Allele,Gene,SYMBOL,Feature,Feature_type,Consequence,"
            "cDNA_position,CDS_position,Protein_position,Amino_acids,Codons,"
            "MANE_SELECT,CANONICAL,AF,gnomADe_AF,SpliceAI_pred,"
            "phyloP100way_vertebrate,phyloP470way_mammalian,phyloP17way_primate,"
            "phastCons100way_vertebrate,phastCons470way_mammalian,phastCons17way_primate,"
            "GERP++_RS,GERP++_NR,GERP_92_mammals"
        )
    ]

    # 2.3 Thực thi Docker
    try:
        subprocess.run(vep_cmd, check=True)
    except subprocess.CalledProcessError as e:
        print(f"\n[LỖI VEP - Chunk {chunk_idx}]")
        print(e.stderr.strip() or "<empty>")
        raise SystemExit("Dừng pipeline do lỗi chạy VEP.")

    # 2.4 Parse kết quả và Merge
    with open(TMP_VEP_OUT) as f:
        lines = [line.strip() for line in f if line.strip() and not line.startswith("##")]
    
    header_line = [l for l in lines if l.startswith("#Uploaded_variation")][0]
    cols = header_line.lstrip("#").split("\t")
    data_lines = [l for l in lines if not l.startswith("#")]
    
    vep_df = pd.read_csv(StringIO("\n".join(data_lines)), sep="\t", names=cols)
    vep_df = vep_df.dropna(subset=["Uploaded_variation"])
    
    # Merge lại với chunk ban đầu (Sử dụng Uploaded_variation từ VEP và Merge_ID của df)
    merged_chunk = df_chunk.merge(
        vep_df, 
        left_on="Merge_ID", 
        right_on="Uploaded_variation", 
        how="left"
    )
    
    # Xóa cột ID tạm
    merged_chunk = merged_chunk.drop(columns=["Merge_ID", "Uploaded_variation"], errors="ignore")

    if 'SpliceAI_pred' in merged_chunk.columns:
        # Tách chuỗi bằng dấu '|' (sẽ tạo ra list gồm 9 phần tử)
        split_sa = merged_chunk['SpliceAI_pred'].str.split('|', expand=True)
        
        # Lấy đúng 4 cột điểm Delta Score (Vị trí index 1, 2, 3, 4)
        if split_sa.shape[1] == 9:
            merged_chunk['SpliceAI_pred_DS_AG'] = pd.to_numeric(split_sa[1], errors='coerce')
            merged_chunk['SpliceAI_pred_DS_AL'] = pd.to_numeric(split_sa[2], errors='coerce')
            merged_chunk['SpliceAI_pred_DS_DG'] = pd.to_numeric(split_sa[3], errors='coerce')
            merged_chunk['SpliceAI_pred_DS_DL'] = pd.to_numeric(split_sa[4], errors='coerce')
            
    return merged_chunk

In [2]:
print(f"[*] Đang tải dữ liệu từ {INPUT_PARQUET}...")
df_full = pd.read_parquet(INPUT_PARQUET)
total_rows = len(df_full)
total_chunks = math.ceil(total_rows / CHUNK_SIZE)
print(f"[*] Tổng số variant: {total_rows:,}. Số lượng chunks: {total_chunks}")

# Kiểm tra cột bắt buộc
required_cols = {"CHROM", "POS", "REF", "ALT"}
if not required_cols.issubset(df_full.columns):
    raise ValueError(f"File Parquet thiếu các cột: {required_cols - set(df_full.columns)}")

[*] Đang tải dữ liệu từ D:/variant_data/clinvar_clean.parquet...
[*] Tổng số variant: 4,137,591. Số lượng chunks: 83


In [3]:
# Định nghĩa file Parquet kết quả cuối cùng
FINAL_OUTPUT_PARQUET = r"D:/variant_data/clinvar_vep_mapping.parquet"

# Xóa file cũ nếu có để tránh ghi đè lỗi
if os.path.exists(FINAL_OUTPUT_PARQUET): 
    os.remove(FINAL_OUTPUT_PARQUET)

# Khởi tạo đối tượng Writer của PyArrow
parquet_writer = None

print("[*] Bắt đầu chạy pipeline cuốn chiếu xuất ra Parquet...")

# Xử lý theo vòng lặp
for i in range(total_chunks):
    start_idx = i * CHUNK_SIZE
    end_idx = min((i + 1) * CHUNK_SIZE, total_rows)
    
    print(f"    -> Đang xử lý Chunk {i+1}/{total_chunks} (Rows: {start_idx} - {end_idx})...")
    
    # Cắt batch
    current_chunk = df_full.iloc[start_idx:end_idx].copy()
    
    # Chạy qua VEP Docker
    annotated_chunk = process_chunk(current_chunk, i+1)
    
    # --- CƠ CHẾ GHI CUỐN CHIẾU VÀO PARQUET ---
    # 1. Chuyển đổi Pandas DataFrame thành PyArrow Table
    table = pa.Table.from_pandas(annotated_chunk)
    
    # 2. Khởi tạo Writer ở chunk đầu tiên (để cố định Schema/kiểu dữ liệu của các cột)
    if parquet_writer is None:
        parquet_writer = pq.ParquetWriter(FINAL_OUTPUT_PARQUET, table.schema, compression='snappy')
    
    # 3. Ghi trực tiếp table của chunk này xuống ổ cứng
    parquet_writer.write_table(table)
    
    # Giải phóng bộ nhớ RAM ngay lập tức
    del annotated_chunk
    del current_chunk

# BẮT BUỘC: Đóng writer sau khi chạy xong tất cả các chunk để đóng file Parquet chuẩn định dạng
if parquet_writer:
    parquet_writer.close()

# Dọn dẹp các file VCF/TXT tạm thời trên ổ cứng
if os.path.exists(TMP_VCF): os.remove(TMP_VCF)
if os.path.exists(TMP_VEP_OUT): os.remove(TMP_VEP_OUT)

print(f"\n[SUCCESS] Pipeline hoàn tất! Kết quả cuối cùng lưu tại: {FINAL_OUTPUT_PARQUET}")

[*] Bắt đầu chạy pipeline cuốn chiếu xuất ra Parquet...
    -> Đang xử lý Chunk 1/83 (Rows: 0 - 50000)...
    -> Đang xử lý Chunk 2/83 (Rows: 50000 - 100000)...
    -> Đang xử lý Chunk 3/83 (Rows: 100000 - 150000)...
    -> Đang xử lý Chunk 4/83 (Rows: 150000 - 200000)...
    -> Đang xử lý Chunk 5/83 (Rows: 200000 - 250000)...
    -> Đang xử lý Chunk 6/83 (Rows: 250000 - 300000)...
    -> Đang xử lý Chunk 7/83 (Rows: 300000 - 350000)...
    -> Đang xử lý Chunk 8/83 (Rows: 350000 - 400000)...
    -> Đang xử lý Chunk 9/83 (Rows: 400000 - 450000)...
    -> Đang xử lý Chunk 10/83 (Rows: 450000 - 500000)...
    -> Đang xử lý Chunk 11/83 (Rows: 500000 - 550000)...
    -> Đang xử lý Chunk 12/83 (Rows: 550000 - 600000)...
    -> Đang xử lý Chunk 13/83 (Rows: 600000 - 650000)...
    -> Đang xử lý Chunk 14/83 (Rows: 650000 - 700000)...
    -> Đang xử lý Chunk 15/83 (Rows: 700000 - 750000)...
    -> Đang xử lý Chunk 16/83 (Rows: 750000 - 800000)...
    -> Đang xử lý Chunk 17/83 (Rows: 800000 - 85

In [1]:
import pandas as pd
final_df = pd.read_parquet(r"D:/variant_data/clinvar_vep_mapping.parquet")

In [2]:
final_df['Consequence'].value_counts()

Consequence
missense_variant                                                                                          2270106
synonymous_variant                                                                                         691731
intron_variant                                                                                             336913
downstream_gene_variant                                                                                    142198
splice_polypyrimidine_tract_variant,intron_variant                                                          97051
upstream_gene_variant                                                                                       93804
stop_gained                                                                                                 83572
missense_variant,splice_region_variant                                                                      75532
splice_region_variant,splice_polypyrimidine_tract_variant,intron_variant    

In [3]:
final_df.isnull().sum()

#AlleleID                           0
Type                                0
Name                                0
GeneID                              0
GeneSymbol                        748
HGNC_ID                          4484
ClinicalSignificance           235824
ClinSigSimple                       0
PhenotypeIDS                    48385
PhenotypeList                     562
Origin                              0
OriginSimple                        0
Assembly                            0
ChromosomeAccession                 0
CHROM                               0
Start                               0
Stop                                0
ReviewStatus                   235824
VariationID                         0
POS                                 0
REF                                 0
ALT                                 0
SomaticClinicalImpact         4136138
ReviewStatusClinicalImpact    4136138
Oncogenicity                  4135773
ReviewStatusOncogenicity      4135773
Location    

In [4]:
import numpy as np

# Danh sách giá trị cần kiểm tra
invalid_values = ["-", "na", "n/a", "NA", "N/A", "NaN", "nan"]

# Thay thế toàn bộ giá trị lỗi trong toàn DataFrame bằng np.nan
final_df = final_df.replace(invalid_values, np.nan)
final_df.isnull().sum()

#AlleleID                           0
Type                                0
Name                                0
GeneID                              0
GeneSymbol                        748
HGNC_ID                          4484
ClinicalSignificance           235824
ClinSigSimple                       0
PhenotypeIDS                    48385
PhenotypeList                     562
Origin                              0
OriginSimple                        0
Assembly                            0
ChromosomeAccession                 0
CHROM                               0
Start                               0
Stop                                0
ReviewStatus                   235824
VariationID                         0
POS                                 0
REF                                 0
ALT                                 0
SomaticClinicalImpact         4136138
ReviewStatusClinicalImpact    4136138
Oncogenicity                  4135773
ReviewStatusOncogenicity      4135773
Location    

In [5]:
final_df = final_df[final_df['Consequence'] == 'missense_variant']
final_df.isnull().sum()

#AlleleID                           0
Type                                0
Name                                0
GeneID                              0
GeneSymbol                          3
HGNC_ID                          1158
ClinicalSignificance            41168
ClinSigSimple                       0
PhenotypeIDS                    18992
PhenotypeList                     364
Origin                              0
OriginSimple                        0
Assembly                            0
ChromosomeAccession                 0
CHROM                               0
Start                               0
Stop                                0
ReviewStatus                    41168
VariationID                         0
POS                                 0
REF                                 0
ALT                                 0
SomaticClinicalImpact         2269037
ReviewStatusClinicalImpact    2269037
Oncogenicity                  2269082
ReviewStatusOncogenicity      2269082
Location    

In [6]:
final_df = final_df.dropna(subset=['ClinicalSignificance', 'Consequence', 'SYMBOL', 'MANE_SELECT', 'CANONICAL'])
final_df.isnull().sum()

#AlleleID                           0
Type                                0
Name                                0
GeneID                              0
GeneSymbol                          0
HGNC_ID                          1130
ClinicalSignificance                0
ClinSigSimple                       0
PhenotypeIDS                    18751
PhenotypeList                     357
Origin                              0
OriginSimple                        0
Assembly                            0
ChromosomeAccession                 0
CHROM                               0
Start                               0
Stop                                0
ReviewStatus                        0
VariationID                         0
POS                                 0
REF                                 0
ALT                                 0
SomaticClinicalImpact         2223203
ReviewStatusClinicalImpact    2223203
Oncogenicity                  2223230
ReviewStatusOncogenicity      2223230
Location    

In [7]:
final_df['ClinicalSignificance'].value_counts().to_string()

'ClinicalSignificance\nUncertain significance                                                    1923272\nConflicting classifications of pathogenicity                               100418\nLikely benign                                                               92780\nLikely pathogenic                                                           30158\nBenign                                                                      27769\nPathogenic                                                                  20641\nBenign/Likely benign                                                        12843\nPathogenic/Likely pathogenic                                                11099\nnot provided                                                                 2879\nother                                                                         463\ndrug response                                                                 361\nno classification for the single variant                        

In [8]:
final_df = final_df[final_df['ClinicalSignificance'].isin(['Likely benign', 'Benign', 'Likely Pathogenic', 'Pathogenic','Benign/Likely benign', 'Pathogenic/Likely pathogenic'])]

In [20]:
final_df.isnull().sum()

#AlleleID                          0
Type                               0
Name                               0
GeneID                             0
GeneSymbol                         0
HGNC_ID                          241
ClinicalSignificance               0
ClinSigSimple                      0
PhenotypeIDS                    4623
PhenotypeList                      2
Origin                             0
OriginSimple                       0
Assembly                           0
ChromosomeAccession                0
CHROM                              0
Start                              0
Stop                               0
ReviewStatus                       0
VariationID                        0
POS                                0
REF                                0
ALT                                0
SomaticClinicalImpact         164875
ReviewStatusClinicalImpact    164875
Oncogenicity                  164933
ReviewStatusOncogenicity      164933
Location                           0
A

In [12]:
final_df.describe()

,#AlleleID,GeneID,ClinSigSimple,Start,Stop,VariationID,POS
count,1.651320e+05,1.651320e+05,165132.000000,1.651320e+05,1.651320e+05,1.651320e+05,1.651320e+05
mean,2.067841e+06,2.945768e+05,0.195183,7.726614e+07,7.726614e+07,2.024406e+06,7.726614e+07
std,1.504207e+06,4.923273e+06,0.396343,5.872247e+07,5.872247e+07,1.443822e+06,5.872247e+07
min,1.505700e+04,-1.000000e+00,0.000000,1.488400e+04,1.488400e+04,1.800000e+01,1.488400e+04
25%,7.017368e+05,3.908000e+03,0.000000,3.208743e+07,3.208743e+07,7.166182e+05,3.208743e+07
50%,2.007666e+06,9.321000e+03,0.000000,6.086525e+07,6.086525e+07,2.079859e+06,6.086525e+07
75%,3.347916e+06,5.768800e+04,0.000000,1.160195e+08,1.160195e+08,3.191229e+06,1.160195e+08
max,4.968200e+06,1.207661e+08,1.000000,2.489181e+08,2.489181e+08,4.857278e+06,2.489181e+08


In [19]:
final_df['AF'].astype(float).describe()

count    54063.000000
mean         0.035934
std          0.128472
min          0.000000
25%          0.000400
50%          0.001800
75%          0.007800
max          1.000000
Name: AF, dtype: float64

In [24]:
final_df['AF'] = final_df['AF'].astype(float)
final_df['AF'][final_df['AF'] == 0].count()

np.int64(71)

In [30]:
final_df[final_df['AF'].isnull() & final_df['gnomADe_AF'].isnull()][['AF', 'gnomADe_AF']]

,AF,gnomADe_AF
1240,NaN,NaN
1525,NaN,NaN
2709,NaN,NaN
2753,NaN,NaN
3886,NaN,NaN
...,...,...
4136620,NaN,NaN
4136639,NaN,NaN
4136645,NaN,NaN
4136648,NaN,NaN


In [37]:
final_df.dropna(subset=['GERP_92_mammals'], how='all', inplace=True)
final_df.dropna(subset=['phyloP470way_mammalian'], how='all', inplace=True)
final_df.dropna(subset=['GERP++_RS'], how='all', inplace=True)
final_df.dropna(subset=['GERP++_NR'], how='all', inplace=True)

In [38]:
final_df.isnull().sum()

#AlleleID                          0
Type                               0
Name                               0
GeneID                             0
GeneSymbol                         0
HGNC_ID                          137
ClinicalSignificance               0
ClinSigSimple                      0
PhenotypeIDS                    3184
PhenotypeList                      0
Origin                             0
OriginSimple                       0
Assembly                           0
ChromosomeAccession                0
CHROM                              0
Start                              0
Stop                               0
ReviewStatus                       0
VariationID                        0
POS                                0
REF                                0
ALT                                0
SomaticClinicalImpact         110921
ReviewStatusClinicalImpact    110921
Oncogenicity                  110967
ReviewStatusOncogenicity      110967
Location                           0
A

In [39]:
final_df.replace({'AF': {np.nan: 0}}, inplace=True)
final_df.replace({'gnomADe_AF': {np.nan: 0}}, inplace=True)

,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,PhenotypeIDS,PhenotypeList,...,SpliceAI_pred,phyloP100way_vertebrate,phyloP470way_mammalian,phyloP17way_primate,phastCons100way_vertebrate,phastCons470way_mammalian,phastCons17way_primate,GERP++_RS,GERP++_NR,GERP_92_mammals
1751,446939,single nucleotide variant,NM_005101.4(ISG15):c.62G>A (p.Ser21Asn),9636,ISG15,HGNC:4053,Benign/Likely benign,0,"MONDO:MONDO:0014502,MedGen:C4015293,OMIM:61612...",Mendelian susceptibility to mycobacterial dise...,...,NaN,-0.676000,-0.213000,-0.552000,0.000000,0.000000,0.026000,1.92,4.25,0.569
1800,389314,single nucleotide variant,NM_005101.4(ISG15):c.248G>A (p.Ser83Asn),9636,ISG15,HGNC:4053,Benign,0,"MedGen:CN169374|MONDO:MONDO:0014502,MedGen:C40...",not specified|Mendelian susceptibility to myco...,...,NaN,-0.396000,-0.065000,-0.731000,0.000000,0.058000,0.523000,-2.36,3.99,-2.78
1866,446981,single nucleotide variant,NM_005101.4(ISG15):c.491G>C (p.Arg164Pro),9636,ISG15,HGNC:4053,Likely benign,0,"MONDO:MONDO:0014502,MedGen:C4015293,OMIM:61612...",Mendelian susceptibility to mycobacterial dise...,...,NaN,-2.665000,-0.684000,-4.089000,0.000000,0.000000,0.000000,-5.24,2.62,-2.13
1879,364282,single nucleotide variant,NM_198576.4(AGRN):c.11G>C (p.Arg4Pro),375790,AGRN,HGNC:329,Benign,0,"MONDO:MONDO:0014052,MedGen:C3808739,OMIM:61512...",Congenital myasthenic syndrome 8|not provided|...,...,NaN,-1.405000,-0.213000,-0.567000,0.000000,0.222000,0.018000,-2.29,2.02,-0.766
2036,446942,single nucleotide variant,NM_198576.4(AGRN):c.494C>T (p.Pro165Leu),375790,AGRN,HGNC:329,Likely benign,0,"MONDO:MONDO:0014052,MedGen:C3808739,OMIM:61512...",Congenital myasthenic syndrome 8|not provided,...,NaN,0.006000,2.166000,0.547000,0.000000,0.036000,0.027000,3.07,4.08,1.02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4134257,2821775,single nucleotide variant,NM_018196.4(TMLHE):c.722G>A (p.Arg241Gln),55217,TMLHE,HGNC:18308,Likely benign,0,MedGen:C3661900,not provided,...,NaN,4.109000,2.889000,-0.309000,0.999000,0.996000,0.984000,2.77,3.64,0.883
4134331,2703929,single nucleotide variant,NM_018196.4(TMLHE):c.149C>A (p.Thr50Asn),55217,TMLHE,HGNC:18308,Likely benign,0,MedGen:CN169374,not specified,...,NaN,0.344000,0.089000,-0.277000,0.620000,0.019000,0.497000,2.76,3.92,-0.0562
4134342,3343405,single nucleotide variant,NM_018196.4(TMLHE):c.58G>A (p.Gly20Arg),55217,TMLHE,HGNC:18308,Likely benign,0,MedGen:CN169374,not specified,...,NaN,-0.898000,-1.000000,0.450000,0.010000,0.000000,0.943000,-4.18,3.91,-2.39
4134688,706348,single nucleotide variant,NM_002186.3(IL9R):c.991G>A (p.Gly331Arg),3581,IL9R,HGNC:6030,Benign,0,MedGen:C3661900,not provided,...,NaN,-0.349000,0.718000,-0.219000,0.000000,0.000000,0.004000,0.252,1.44,-1.46


In [40]:
final_df.isnull().sum()

#AlleleID                          0
Type                               0
Name                               0
GeneID                             0
GeneSymbol                         0
HGNC_ID                          137
ClinicalSignificance               0
ClinSigSimple                      0
PhenotypeIDS                    3184
PhenotypeList                      0
Origin                             0
OriginSimple                       0
Assembly                           0
ChromosomeAccession                0
CHROM                              0
Start                              0
Stop                               0
ReviewStatus                       0
VariationID                        0
POS                                0
REF                                0
ALT                                0
SomaticClinicalImpact         110921
ReviewStatusClinicalImpact    110921
Oncogenicity                  110967
ReviewStatusOncogenicity      110967
Location                           0
A

In [47]:
final_df[['gnomADe_AF', 'phyloP100way_vertebrate', 'phyloP470way_mammalian', 'phyloP17way_primate', 'phastCons100way_vertebrate', 'phastCons470way_mammalian', 'phastCons17way_primate', 'GERP++_RS', 'GERP++_NR', 'GERP_92_mammals']] = final_df[['gnomADe_AF', 'phyloP100way_vertebrate', 'phyloP470way_mammalian', 'phyloP17way_primate', 'phastCons100way_vertebrate', 'phastCons470way_mammalian', 'phastCons17way_primate', 'GERP++_RS', 'GERP++_NR', 'GERP_92_mammals']].astype(float)

In [48]:
final_df.describe()

,#AlleleID,GeneID,ClinSigSimple,Start,Stop,VariationID,POS,AF,gnomADe_AF,phyloP100way_vertebrate,phyloP470way_mammalian,phyloP17way_primate,phastCons100way_vertebrate,phastCons470way_mammalian,phastCons17way_primate,GERP++_RS,GERP++_NR,GERP_92_mammals
count,1.110830e+05,1.110830e+05,111083.000000,1.110830e+05,1.110830e+05,1.110830e+05,1.110830e+05,111083.000000,111083.000000,111083.000000,111083.000000,111083.000000,111083.000000,111083.000000,111083.000000,111083.000000,111083.000000,111083.000000
mean,2.088946e+06,2.318603e+05,0.161726,7.651980e+07,7.651980e+07,2.045012e+06,7.651980e+07,0.012976,0.012599,2.500176,3.080988,0.108484,0.598214,0.605744,0.610337,1.713919,5.058315,-0.508660
std,1.500283e+06,4.247310e+06,0.368201,5.787842e+07,5.787842e+07,1.439267e+06,5.787842e+07,0.079974,0.081924,3.211944,4.824221,0.800358,0.460927,0.470142,0.412271,4.013487,0.846766,2.491854
min,1.505700e+04,-1.000000e+00,0.000000,4.923400e+04,4.923400e+04,1.800000e+01,4.923400e+04,0.000000,0.000000,-20.000000,-20.000000,-8.801000,0.000000,0.000000,0.000000,-12.300000,0.046500,-16.300000
25%,7.069790e+05,4.010000e+03,0.000000,3.235657e+07,3.235657e+07,7.237090e+05,3.235657e+07,0.000000,0.000001,0.121000,0.039000,-0.190000,0.004000,0.000000,0.108000,-0.248500,4.710000,-2.720000
50%,2.045311e+06,9.758000e+03,0.000000,6.151805e+07,6.151805e+07,2.118886e+06,6.151805e+07,0.000000,0.000017,1.499000,2.314000,0.589000,0.969000,0.996000,0.838000,3.020000,5.260000,-0.091100
75%,3.356700e+06,5.772200e+04,0.000000,1.124504e+08,1.124504e+08,3.199986e+06,1.124504e+08,0.000500,0.000200,4.632000,7.061000,0.665000,1.000000,1.000000,0.989000,4.820000,5.640000,1.660000
max,4.968200e+06,1.124414e+08,1.000000,2.488564e+08,2.488564e+08,4.857278e+06,2.488564e+08,1.000000,1.000000,10.003000,11.936000,0.756000,1.000000,1.000000,1.000000,6.170000,6.170000,10.300000


In [49]:
final_df.to_parquet(r'D:/variant_data/clinvar_vep_mapping_final.parquet', index=False)

In [50]:
pd.read_parquet(r'D:/variant_data/clinvar_vep_mapping_final.parquet').head()

,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,PhenotypeIDS,PhenotypeList,...,SpliceAI_pred,phyloP100way_vertebrate,phyloP470way_mammalian,phyloP17way_primate,phastCons100way_vertebrate,phastCons470way_mammalian,phastCons17way_primate,GERP++_RS,GERP++_NR,GERP_92_mammals
0,446939,single nucleotide variant,NM_005101.4(ISG15):c.62G>A (p.Ser21Asn),9636,ISG15,HGNC:4053,Benign/Likely benign,0,"MONDO:MONDO:0014502,MedGen:C4015293,OMIM:61612...",Mendelian susceptibility to mycobacterial dise...,...,NaN,-0.676,-0.213,-0.552,0.0,0.000,0.026,1.92,4.25,0.569
1,389314,single nucleotide variant,NM_005101.4(ISG15):c.248G>A (p.Ser83Asn),9636,ISG15,HGNC:4053,Benign,0,"MedGen:CN169374|MONDO:MONDO:0014502,MedGen:C40...",not specified|Mendelian susceptibility to myco...,...,NaN,-0.396,-0.065,-0.731,0.0,0.058,0.523,-2.36,3.99,-2.780
2,446981,single nucleotide variant,NM_005101.4(ISG15):c.491G>C (p.Arg164Pro),9636,ISG15,HGNC:4053,Likely benign,0,"MONDO:MONDO:0014502,MedGen:C4015293,OMIM:61612...",Mendelian susceptibility to mycobacterial dise...,...,NaN,-2.665,-0.684,-4.089,0.0,0.000,0.000,-5.24,2.62,-2.130
3,364282,single nucleotide variant,NM_198576.4(AGRN):c.11G>C (p.Arg4Pro),375790,AGRN,HGNC:329,Benign,0,"MONDO:MONDO:0014052,MedGen:C3808739,OMIM:61512...",Congenital myasthenic syndrome 8|not provided|...,...,NaN,-1.405,-0.213,-0.567,0.0,0.222,0.018,-2.29,2.02,-0.766
4,446942,single nucleotide variant,NM_198576.4(AGRN):c.494C>T (p.Pro165Leu),375790,AGRN,HGNC:329,Likely benign,0,"MONDO:MONDO:0014052,MedGen:C3808739,OMIM:61512...",Congenital myasthenic syndrome 8|not provided,...,NaN,0.006,2.166,0.547,0.0,0.036,0.027,3.07,4.08,1.020
